# 03 — Data Cleaning & Transformation

**Tujuan notebook ini (Fase 3 roadmap):**
- 3.1 Cleaning — handle missing values, duplicates, tipe data, tanggal invalid
- 3.2 Date Transformation — `delivery_days`, dst (dengan penanganan missing yang benar, BUKAN `fillna(0)`)
- 3.3 Business Metrics — `transaction_value` (bukan "revenue")
- 3.4 Geolocation Cleaning — agregasi per zip code prefix

> Hasil notebook ini disimpan sebagai DataFrame cleaned di memory, dipakai lagi di notebook `04_analytical_tables.ipynb`. Raw collection di MongoDB **tidak diubah** (sesuai Fase 0 #1).


In [8]:
import pandas as pd
import numpy as np
from pymongo import MongoClient

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "olist_db"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

COLLECTIONS = [
    "customers_raw", "orders_raw", "order_items_raw", "payments_raw",
    "reviews_raw", "products_raw", "sellers_raw", "geolocation_raw",
    "category_translation_raw",
]

dfs = {}
for col_name in COLLECTIONS:
    dfs[col_name] = pd.DataFrame(list(db[col_name].find({}, {"_id": 0})))

print("Semua collection ter-load ke DataFrame.")


Semua collection ter-load ke DataFrame.


---
## 3.1 Cleaning

Cek dan tangani: missing values, duplicates, tipe data, tanggal invalid.


In [9]:
orders = dfs["orders_raw"].copy()

# Konversi kolom tanggal ke datetime (invalid parsing otomatis jadi NaT, bukan error)
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

print("Missing value per kolom tanggal setelah konversi:")
print(orders[date_cols].isnull().sum())

print(f"\nDuplicate order_id: {orders['order_id'].duplicated().sum()}")


Missing value per kolom tanggal setelah konversi:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Duplicate order_id: 0


In [10]:
# Cek order_status untuk order yang order_delivered_customer_date-nya kosong
# (ini WAJAR untuk order yang belum/tidak delivered, bukan data error)
missing_delivered = orders[orders["order_delivered_customer_date"].isnull()]
print("Distribusi order_status untuk order dengan order_delivered_customer_date kosong:")
print(missing_delivered["order_status"].value_counts())
print(f"\nTotal: {len(missing_delivered)} dari {len(orders)} order ({len(missing_delivered)/len(orders)*100:.2f}%)")


Distribusi order_status untuk order dengan order_delivered_customer_date kosong:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

Total: 2965 dari 99441 order (2.98%)


**Interpretasi:** kalau mayoritas order dengan `order_delivered_customer_date` kosong itu berstatus `canceled` atau `unavailable` (bukan `delivered`), berarti missing value ini memang representasi kondisi bisnis yang valid — bukan data quality issue yang perlu di-drop atau diisi paksa.


---
## 3.2 Date Transformation

`delivery_days` dan turunannya — **dengan penanganan missing yang benar**.

> ⚠️ Jangan `fillna(0)` — order yang belum/tidak delivered punya `order_delivered_customer_date` kosong, dan `0` di situ akan salah diartikan sebagai "delivered instan". Missing delivery ≠ zero delivery days.


In [11]:
orders["order_year"] = orders["order_purchase_timestamp"].dt.year
orders["order_month"] = orders["order_purchase_timestamp"].dt.month
orders["order_week"] = orders["order_purchase_timestamp"].dt.isocalendar().week
orders["day_of_week"] = orders["order_purchase_timestamp"].dt.day_name()

# delivery_days: biarkan NaN kalau order_delivered_customer_date kosong (JANGAN fillna(0))
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

# estimated_delivery_gap: negatif = lebih cepat dari estimasi, positif = telat
orders["estimated_delivery_gap"] = (
    orders["order_estimated_delivery_date"] - orders["order_delivered_customer_date"]
).dt.days

# Flag eksplisit, supaya "missing" dan "delivered" bisa dibedakan jelas di analisis manapun
orders["is_delivered"] = orders["order_delivered_customer_date"].notnull()

print(f"delivery_days -- missing: {orders['delivery_days'].isnull().sum()}, "
      f"terisi: {orders['delivery_days'].notnull().sum()}")
print(f"\nStatistik delivery_days (HANYA dari yang terisi, bukan termasuk NaN):")
print(orders["delivery_days"].describe())


delivery_days -- missing: 2965, terisi: 96476

Statistik delivery_days (HANYA dari yang terisi, bukan termasuk NaN):
count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64


---
## 3.3 Business Metrics

`transaction_value` — bukan "revenue" (dataset tidak menyediakan cost/refund/profit, jadi ini proxy dari `payment_value`).


In [12]:
payments = dfs["payments_raw"].copy()

# Transaction value per order (sum semua payment record, karena bisa cicilan/lebih dari 1 payment per order)
transaction_value_per_order = payments.groupby("order_id")["payment_value"].sum().rename("transaction_value")

order_items = dfs["order_items_raw"].copy()
items_per_order = order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    freight_value=("freight_value", "sum"),
    price_value=("price", "sum"),
).reset_index()

orders = orders.merge(transaction_value_per_order, on="order_id", how="left")
orders = orders.merge(items_per_order, on="order_id", how="left")

orders["freight_ratio"] = orders["freight_value"] / orders["price_value"]

print(f"Total transaction value (proxy, sum payment_value semua order): "
      f"{orders['transaction_value'].sum():,.2f}")
print(f"Average Order Value: {orders['transaction_value'].mean():,.2f}")
print(f"\nOrder tanpa payment record (transaction_value NaN): {orders['transaction_value'].isnull().sum()}")


Total transaction value (proxy, sum payment_value semua order): 16,008,872.12
Average Order Value: 160.99

Order tanpa payment record (transaction_value NaN): 1


> **Catatan dokumentasi (untuk `docs/methodology.md` nanti):**
> *"Payment value is used as a transaction-value proxy because the dataset does not provide accounting-level revenue, cost, refund, or profit information."*


---
## 3.4 Geolocation Cleaning

`geolocation_raw` punya banyak baris lat/long per zip code prefix — agregasi dulu (median) sebelum dipakai join, supaya tidak menyebabkan row multiplication.


In [13]:
geo = dfs["geolocation_raw"].copy()

print(f"Baris geolocation sebelum agregasi: {len(geo)}")
print(f"Unique zip_code_prefix: {geo['geolocation_zip_code_prefix'].nunique()}")

geo_agg = geo.groupby("geolocation_zip_code_prefix").agg(
    geolocation_lat=("geolocation_lat", "median"),
    geolocation_lng=("geolocation_lng", "median"),
    geolocation_city=("geolocation_city", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    geolocation_state=("geolocation_state", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
).reset_index()

print(f"\nBaris geolocation SETELAH agregasi: {len(geo_agg)}")
print(f"-> Sekarang 1 row = 1 zip_code_prefix, aman untuk dipakai join nanti (Fase 4).")

geo_agg.head()


Baris geolocation sebelum agregasi: 1000163
Unique zip_code_prefix: 19015

Baris geolocation SETELAH agregasi: 19015
-> Sekarang 1 row = 1 zip_code_prefix, aman untuk dipakai join nanti (Fase 4).


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550381,-46.634027,sao paulo,SP
1,1002,-23.548551,-46.635072,sao paulo,SP
2,1003,-23.548977,-46.635313,sao paulo,SP
3,1004,-23.549535,-46.634771,sao paulo,SP
4,1005,-23.549612,-46.636532,sao paulo,SP


---
## Ringkasan Hasil Cleaning (dipakai lagi di notebook 04)

DataFrame berikut sudah siap dipakai untuk membangun analytical tables:
- `orders` — sudah ada `delivery_days`, `is_delivered`, `transaction_value`, `freight_ratio`, dst.
- `geo_agg` — geolocation yang sudah diagregasi per zip prefix.
- `dfs["customers_raw"]`, `dfs["products_raw"]`, dst. — dipakai apa adanya (belum butuh transformasi tambahan di titik ini).


In [14]:
print("orders shape (setelah transformasi):", orders.shape)
print("\nKolom baru yang ditambahkan:")
new_cols = [
    "order_year", "order_month", "order_week", "day_of_week",
    "delivery_days", "estimated_delivery_gap", "is_delivered",
    "transaction_value", "n_items", "freight_value", "price_value", "freight_ratio",
]
print(new_cols)

orders[["order_id", "order_status", "is_delivered", "delivery_days", "transaction_value"]].head()


orders shape (setelah transformasi): (99441, 20)

Kolom baru yang ditambahkan:
['order_year', 'order_month', 'order_week', 'day_of_week', 'delivery_days', 'estimated_delivery_gap', 'is_delivered', 'transaction_value', 'n_items', 'freight_value', 'price_value', 'freight_ratio']


,order_id,order_status,is_delivered,delivery_days,transaction_value
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,True,8.0,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,True,13.0,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,True,9.0,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,True,13.0,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,True,2.0,28.62


---
## Definition of Done (Fase 3)

- [ ] Missing values ditangani dengan tepat (bukan `fillna(0)` sembarangan untuk `delivery_days`)
- [ ] Date transformation lengkap (`order_year`, `delivery_days`, `estimated_delivery_gap`, `is_delivered`)
- [ ] Business metrics pakai istilah `transaction_value`, bukan "revenue" tanpa kualifikasi
- [ ] Geolocation sudah diagregasi per zip prefix (1 row = 1 prefix)

**Lanjut ke:** `04_analytical_tables.ipynb`
